In [3]:
import pandas as pd
import geopandas as gpd
import getpass
from pathlib import Path

user = getpass.getuser()
PROJECT_ROOT = Path(f"/Users/{user}/Final_Lawyer_Git July10")

# MSAs
msa_path = PROJECT_ROOT / "Data/Geography/CBSA_shapefile_2025/tl_2025_us_cbsa.shp"
gdf_msa = gpd.read_file(msa_path)
gdf_msa = gdf_msa[gdf_msa["LSAD"] == "M1"][["GEOID", "NAME"]].copy()
gdf_msa[["City", "State"]] = gdf_msa["NAME"].str.rsplit(",", n=1, expand=True)
gdf_msa["City"] = gdf_msa["City"].str.split("-", n=1).str[0].str.strip()
gdf_msa["State"] = gdf_msa["State"].str.strip().str[:2]
gdf_msa.rename(columns={"GEOID": "AREA"}, inplace=True)

# Lawyers
lawyers_path = PROJECT_ROOT / "Data/BrightData_Lawyers/BrightData_Lawyers_master_normalized_1overN.csv"
lawyers_data = pd.read_csv(lawyers_path)
lawyers_data.rename(columns={
    "CBSA": "AREA",
    "Real Estate Law_normalized_1overN_count": "Real Estate"
}, inplace=True)
lawyers_data = lawyers_data[["AREA", "Real Estate"]]
lawyers_data["AREA"] = lawyers_data["AREA"].astype("Int64").astype(str)
lawyers_data = lawyers_data.dropna(subset=["Real Estate"])

# Zillow transaction values
real_estate_dir = PROJECT_ROOT / "Data/Proxies/Real Estate"
ttv_data = pd.read_csv(real_estate_dir / "Metro_total_transaction_value_now_uc_sfrcondo_month.csv")
ttv_data = ttv_data[ttv_data["RegionType"] == "msa"].copy()
ttv_data[["City", "State"]] = ttv_data["RegionName"].str.rsplit(",", n=1, expand=True)
ttv_data["City"] = ttv_data["City"].str.strip()
ttv_data["State"] = ttv_data["State"].str.strip().str[:2]

months_2024 = [column for column in ttv_data.columns if column.startswith("2024-")]
ttv_data = ttv_data.dropna(subset=months_2024)
ttv_data["Total_Transaction_Value_2024"] = ttv_data[months_2024].sum(axis=1)
ttv_data = ttv_data.merge(gdf_msa[["AREA", "City", "State"]], on=["City", "State"], how="left")

# Zillow uses shorter names for these three metros
ttv_data["AREA"] = ttv_data["AREA"].fillna(ttv_data["RegionName"].map({
    "Poughkeepsie, NY": "28880", "Louisville, KY": "31140", "The Villages, FL": "48680"
}))

ttv_data = ttv_data.dropna(subset=["AREA"])
ttv_data = ttv_data.groupby("AREA", as_index=False)["Total_Transaction_Value_2024"].sum()
ttv_data["Total_Transaction_Value_2024_millions"] = ttv_data["Total_Transaction_Value_2024"] / 1_000_000

real_estate_data = lawyers_data.merge(ttv_data, on="AREA", how="inner")
real_estate_data = real_estate_data[["AREA", "Real Estate", "Total_Transaction_Value_2024", "Total_Transaction_Value_2024_millions"]]
real_estate_data = real_estate_data.sort_values("AREA").reset_index(drop=True)
real_estate_data.to_csv(real_estate_dir / "Real_Estate_Proxy_Normalized.csv", index=False)